In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the CSV file
import os
import pandas as pd
food_path = os.path.join(path, 'q1-ka-ai-2026.csv')
food_data = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')


In [ ]:
# Task 2: Write your code here:
food_data.head()


In [ ]:
# Task 3: Write your code here:
food_data.info()

In [ ]:
# Task 4: Write your code here:
food_data.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(food_data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
# I will now do the EDA to clean the data and make it ready for the training
clean_df = food_data.drop(columns='Order_ID')
clean_df.head()


In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(clean_df)

clean_df['Traffic_Level'] = clean_df['Traffic_Level'].fillna('none')
clean_df['Weather'] = clean_df['Weather'].fillna('none')
clean_df['Time_of_Day'] = clean_df['Time_of_Day'].fillna('none')
clean_df['Courier_Experience_yrs'] = clean_df['Courier_Experience_yrs'].fillna(value=clean_df["Delivery_Time"].median)
clean_df['Delivery_Time'] = clean_df['Delivery_Time'].fillna(value=clean_df["Delivery_Time"].median)


In [ ]:
# Task 3: Write your code here:
#Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(clean_df)

clean_df.info()



In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder


# I need to convert these two columns to float values!!
clean_df['Courier_Experience_yrs'] = clean_df['Courier_Experience_yrs'].values=clean_df["Courier_Experience_yrs"].median
clean_df['Delivery_Time'] = clean_df['Delivery_Time'].values=clean_df["Delivery_Time"].median
# Do we have categorical columns?
categorical_cols = clean_df.select_dtypes(include=["object"]).columns.drop('Delivery_Time','Courier_Experience_yrs')

print("Categorical Columns:", list(categorical_cols))




In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = clean_df.select_dtypes(include=["int64", "float64"]).columns.drop('Delivery_Time')  # DON'T SCALE THE TARGET

scaler = StandardScaler()
clean_df[numerical_cols] = scaler.fit_transform(clean_df[numerical_cols])
clean_df.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = clean_df.drop("Delivery_Time", axis=1).astype(float)
y = clean_df['Delivery_Time'].astype(float)
print(y)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from tqdm import tqdm
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits= n_splits, shuffle=True, random_state=42, Stratify= y) # we need to use stratify to balance the data
# Storage for linear regression results for each fold
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

In [ ]:
# Task 1: Write your code here:
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:
print("Linear Regression Results")
print(f"  Average MSE: {np.mean(lr_mse):.4f}")
print(f"  Average RMSE: {np.mean(lr_rmse):.4f}")
print(f"  Average R2:  {np.mean(lr_r2):.4f}")

In [ ]:
# Task Bonus: Write your code here: